# **Task 03 – Data Preprocessing and Feature Engineering**
### **SmartCare Hospital AI Dataset | Option C – Disease Risk Classification**

**Target:** `disease_risk_level` (Low=0, Medium=1, High=2)

**Leakage-safety principle:** any preprocessing step that learns a value from
the data (imputation statistics, correlation-based feature selection,
scaling parameters) is fit on the training set only, after the train/test
split, and then applied to the test set unchanged. Deterministic corrections
that don't depend on dataset statistics (physiological validity fixes,
fixed-threshold feature engineering) may be applied before the split.

**Pipeline order:** Load → Quality Check → Deterministic Missing-Value Step →
Duplicates → Outlier Investigation → Data Cleaning → Target/Predictor
Separation → Stratified Split → Train-Only Imputation → Encoding → Feature
Engineering → Train-Only Feature Selection → Train-Only Scaling → Verify →
Save.

## **Section 1 — Load and Preserve Raw Data**
`df_raw` is kept untouched for the whole notebook; all work happens on `df`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

PROJECT_ROOT = Path.cwd().parent
df_raw = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'smartcare_ai_dataset_1000.csv')
df = df_raw.copy()

print(f"Shape: {df.shape}")
df.head()

Shape: (1000, 33)


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,missed_previous_appointments,appointment_status,admitted,room_type,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,0,Completed,0,NaN,0,1,127,75,117,211,26.1,0,3,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,1,Completed,0,NaN,0,0,130,73,136,173,32.8,0,1,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,1,No-Show,0,NaN,0,1,141,64,90,176,29.4,1,0,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,0,Completed,0,NaN,0,0,124,82,126,189,24.9,2,1,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,0,Scheduled,0,NaN,0,1,119,81,65,195,27.0,2,0,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


## **Section 2 — Initial Data-Quality Checks**
Recap of Task 02 findings, used as the checklist for this notebook.

In [2]:
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nFull-row duplicates: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df['disease_risk_level'].value_counts(normalize=True).round(3) * 100}")

Missing values:
room_type    906
dtype: int64

Full-row duplicates: 0

Target distribution:
disease_risk_level
Medium    46.9
High      40.0
Low       13.1
Name: proportion, dtype: float64


## **Section 3 — Missing-Value Handling (Deterministic Portion)**
`room_type` missingness has two causes (see Task 02):
- **Non-admitted patients**: missing because no room was ever assigned —
  structural, not an error. Resolved here with an explicit `"Not Admitted"`
  category — this is a fixed rule, not a data-dependent statistic, so it's
  safe to apply before the split.
- **Admitted patients with missing `room_type`**: a genuine gap. Filling
  this requires a statistic (mode) computed from the data, so it is
  **deferred to Section 10 (post-split, train-only)** to avoid leakage.

In [3]:
mask_not_admitted = (df['admitted'] == 0) & (df['room_type'].isnull())
df.loc[mask_not_admitted, 'room_type'] = 'Not Admitted'
print(f"Rows set to 'Not Admitted': {mask_not_admitted.sum()}")
print(f"Remaining missing room_type (admitted, deferred to train-only step): {df['room_type'].isnull().sum()}")

Rows set to 'Not Admitted': 670
Remaining missing room_type (admitted, deferred to train-only step): 236


## **Section 4 — Duplicate Record Detection**
Checked at three levels; repeated `patient_id` alone is expected (same
patient, multiple visits) and is not an error.

In [4]:
full_duplicates = df.duplicated().sum()
duplicate_record_ids = df['record_id'].duplicated().sum()
duplicate_patient_ids = df['patient_id'].duplicated().sum()

print(f"Full-row duplicates: {full_duplicates}")
print(f"Duplicate record_id: {duplicate_record_ids}")
print(f"Repeated patient_id (expected, not an error): {duplicate_patient_ids}")

if full_duplicates > 0:
    df = df.drop_duplicates()
    print(f"Dropped {full_duplicates} rows. New shape: {df.shape}")
else:
    print("No action required.")

Full-row duplicates: 0
Duplicate record_id: 0
Repeated patient_id (expected, not an error): 0
No action required.


## **Section 5 — Outlier Investigation**
IQR bounds on clinical predictors, for documentation only — no rows are
removed or capped here. Extreme clinical values (e.g. very high cholesterol)
are plausibly the actual signal for "High" risk, not noise; this decision
is revisited with train-only evidence in Section 13 (Feature Selection).

Billing rate consistency is checked separately as an integrity check
(billing fields aren't used as predictors for this target).

In [5]:
def iqr_outlier_report(series, col_name):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"{col_name:20s} | bounds=({lower:.1f}, {upper:.1f}) | outliers={len(outliers)}")

clinical_cols = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi']
for col in clinical_cols:
    iqr_outlier_report(df[col], col)

age                  | bounds=(-3.0, 93.0) | outliers=0
systolic_bp          | bounds=(84.0, 172.0) | outliers=3
diastolic_bp         | bounds=(51.0, 107.0) | outliers=3
blood_sugar_mg_dl    | bounds=(44.5, 184.5) | outliers=10
cholesterol_mg_dl    | bounds=(105.0, 305.0) | outliers=6
bmi                  | bounds=(14.3, 36.7) | outliers=9


In [6]:
# Billing rate integrity check (not used as a predictor)
paid_rows = df[df['room_charge_lkr'] > 0].copy()
paid_rows['rate_per_day'] = paid_rows['room_charge_lkr'] / paid_rows['length_of_stay_days']
print(paid_rows.groupby('room_type')['rate_per_day'].std())
print("Std ≈ 0 per room_type confirms a fixed daily rate — not a data error.")

room_type
General Ward    0.0
ICU             0.0
Private Room    0.0
Name: rate_per_day, dtype: float64
Std ≈ 0 per room_type confirms a fixed daily rate — not a data error.


## **Section 6 — Data Cleaning**
Two deterministic, physiologically-motivated corrections. No rows dropped.

| Issue | Rows | Action | Note |
|---|---|---|---|
| `missed_previous_appointments > previous_appointments` | 16 | Cap at `previous_appointments` | Impossible count |
| `systolic_bp <= diastolic_bp` | 3 | Swap values | **Treated as a probable data-entry transposition — this is an assumption, not a verified fact** |
| `treatments_count == 0` but `medicine_charge_lkr > 0` | 212 | Documented, not corrected | Billing field, out of scope for this target |

In [7]:
mask_bad_missed = df['missed_previous_appointments'] > df['previous_appointments']
df.loc[mask_bad_missed, 'missed_previous_appointments'] = df.loc[mask_bad_missed, 'previous_appointments']
print(f"Capped {mask_bad_missed.sum()} rows.")

mask_bad_bp = df['systolic_bp'] <= df['diastolic_bp']
print(f"Rows with systolic <= diastolic (assumed transposition): {mask_bad_bp.sum()}")
df.loc[mask_bad_bp, ['systolic_bp', 'diastolic_bp']] = df.loc[mask_bad_bp, ['diastolic_bp', 'systolic_bp']].values

mask_billing = (df['treatments_count'] == 0) & (df['medicine_charge_lkr'] > 0)
print(f"Billing inconsistency rows (documented only): {mask_billing.sum()}")
print(f"Rows lost this section: 0 (all {len(df)} rows retained)")

Capped 16 rows.
Rows with systolic <= diastolic (assumed transposition): 3
Billing inconsistency rows (documented only): 212
Rows lost this section: 0 (all 1000 rows retained)


## **Section 7 — Target/Predictor Separation**
Drop identifiers, the two alternative AI targets, and `appointment_date`
(not usable as a raw predictor). Encode the target ordinally — the classes
have a genuine order (Low < Medium < High), unlike the nominal predictors.
This mapping is fixed, not data-dependent, so it's safe pre-split.

In [8]:
df['disease_risk_level_label'] = df['disease_risk_level']
risk_order = {'Low': 0, 'Medium': 1, 'High': 2}
df['disease_risk_level'] = df['disease_risk_level_label'].map(risk_order)

drop_structural = ['record_id', 'patient_id', 'no_show', 'readmitted_30_days',
                    'appointment_date', 'disease_risk_level_label']

X = df.drop(columns=drop_structural + ['disease_risk_level'])
y = df['disease_risk_level']

print(f"X shape: {X.shape}")
print(f"Dropped (identifiers/alt-targets/admin): {drop_structural}")

X shape: (1000, 27)
Dropped (identifiers/alt-targets/admin): ['record_id', 'patient_id', 'no_show', 'readmitted_30_days', 'appointment_date', 'disease_risk_level_label']


## **Section 8 — Stratified Train/Test Split**
Stratified on `y` to preserve class proportions (Low is only ~13% of the
data). Everything data-dependent from this point on is fit on `X_train`
only.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train class %:\n", y_train.value_counts(normalize=True).round(3) * 100)
print("Test class %:\n", y_test.value_counts(normalize=True).round(3) * 100)

Train: (800, 27), Test: (200, 27)
Train class %:
 disease_risk_level
1    46.9
2    40.0
0    13.1
Name: proportion, dtype: float64
Test class %:
 disease_risk_level
1    47.0
2    40.0
0    13.0
Name: proportion, dtype: float64


## **Section 9 — Training-Only Missing-Value Imputation**
The mode `room_type` among admitted patients is computed from `X_train`
only, then applied to fill the remaining nulls in **both** `X_train` and
`X_test`. This is the leakage fix for the imputation step flagged earlier.

In [10]:
admitted_mode = X_train.loc[(X_train['admitted'] == 1) & (X_train['room_type'].notnull()), 'room_type'].mode()[0]
print(f"Mode (from X_train only): '{admitted_mode}'")

for split_df in (X_train, X_test):
    split_df['room_type'] = split_df['room_type'].fillna(admitted_mode)

print(f"Remaining nulls — train: {X_train.isnull().sum().sum()}, test: {X_test.isnull().sum().sum()}")

Mode (from X_train only): 'General Ward'
Remaining nulls — train: 0, test: 0


## **Section 10 — Categorical Encoding**
One-hot encoding for nominal predictors (no natural order). Category
columns are derived from `X_train` only; `X_test` is aligned to the same
column set (any category unseen in training gets all-zero encoding, and
vice versa) so both sets have identical feature columns.

In [11]:
nominal_cols = ['gender', 'blood_group', 'department', 'diagnosis', 'room_type',
                 'payment_method', 'payment_status', 'appointment_status']

def onehot_align(train_df, test_df, cols):
    train_enc = pd.get_dummies(train_df, columns=cols)
    test_enc = pd.get_dummies(test_df, columns=cols).reindex(columns=train_enc.columns, fill_value=0)
    return train_enc, test_enc

X_train, X_test = onehot_align(X_train, X_test, nominal_cols)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (800, 61), Test shape: (200, 61)


## **Section 11 — Feature Engineering**
Fixed clinical-threshold bands (WHO/AHA-style) applied identically to train
and test — thresholds are constants, not learned from data, so this is
leakage-safe. `health_burden_score` is a simple arithmetic composite.

In [12]:
def bp_category(row):
    s, d = row['systolic_bp'], row['diastolic_bp']
    if s < 120 and d < 80: return 'Normal'
    elif s < 130 and d < 80: return 'Elevated'
    return 'Hypertensive'

def bmi_category(bmi):
    if bmi < 18.5: return 'Underweight'
    elif bmi < 25: return 'Normal'
    elif bmi < 30: return 'Overweight'
    return 'Obese'

def blood_sugar_category(bs):
    if bs < 100: return 'Normal'
    elif bs < 126: return 'Prediabetic'
    return 'Diabetic'

def age_group(age):
    if age < 13: return 'Child'
    elif age < 30: return 'Young Adult'
    elif age < 60: return 'Adult'
    return 'Senior'

eng_nominal = ['bp_category', 'bmi_category', 'blood_sugar_category', 'age_group']

for split_df in (X_train, X_test):
    split_df['bp_category'] = split_df.apply(bp_category, axis=1)
    split_df['bmi_category'] = split_df['bmi'].apply(bmi_category)
    split_df['blood_sugar_category'] = split_df['blood_sugar_mg_dl'].apply(blood_sugar_category)
    split_df['age_group'] = split_df['age'].apply(age_group)
    split_df['health_burden_score'] = split_df['previous_admissions'] + split_df['previous_appointments']

X_train, X_test = onehot_align(X_train, X_test, eng_nominal)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (800, 76), Test shape: (200, 76)


## **Section 12 — Training-Only Feature Selection**
Correlation against `y_train` (not the full dataset) determines which
billing/operational numeric columns to drop. Low correlation is described
as **weak linear association**, not "no predictive value" — a linear
correlation coefficient can't rule out non-linear relationships.

In [13]:
numeric_for_corr = ['waiting_days', 'previous_appointments', 'missed_previous_appointments',
                     'previous_admissions', 'lab_tests_count', 'treatments_count',
                     'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr',
                     'medicine_charge_lkr', 'total_bill_lkr']

corr_train = pd.concat([X_train[numeric_for_corr], y_train], axis=1).corr()['disease_risk_level'].drop('disease_risk_level')
print("Correlation with y_train:\n", corr_train.sort_values())

drop_weak = corr_train[corr_train.abs() < 0.1].index.tolist()
X_train = X_train.drop(columns=drop_weak)
X_test = X_test.drop(columns=drop_weak)
print(f"\nDropped (weak linear association, |r|<0.1, train-only): {drop_weak}")
print(f"Shape — train: {X_train.shape}, test: {X_test.shape}")

Correlation with y_train:
 consultation_fee_lkr           -0.021997
waiting_days                    0.000410
lab_charge_lkr                  0.007767
lab_tests_count                 0.008589
treatments_count                0.011607
room_charge_lkr                 0.020133
total_bill_lkr                  0.027186
medicine_charge_lkr             0.028927
missed_previous_appointments    0.032974
previous_appointments           0.044080
previous_admissions             0.126729
Name: disease_risk_level, dtype: float64

Dropped (weak linear association, |r|<0.1, train-only): ['waiting_days', 'previous_appointments', 'missed_previous_appointments', 'lab_tests_count', 'treatments_count', 'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr', 'total_bill_lkr']
Shape — train: (800, 66), test: (200, 66)


## **Section 13 — Training-Only Feature Scaling**
`StandardScaler` fit on `X_train` only, then used to transform both sets —
the test set never influences the fitted mean/std.

In [14]:
numeric_cols_to_scale = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl',
                          'cholesterol_mg_dl', 'bmi', 'previous_admissions', 'health_burden_score']

scaler = StandardScaler()
X_train[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test[numeric_cols_to_scale] = scaler.transform(X_test[numeric_cols_to_scale])

X_train[numeric_cols_to_scale].describe().round(2)

,age,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,previous_admissions,health_burden_score
count,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00
mean,0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-2.45,-2.83,-2.89,-1.88,-2.84,-2.73,-0.90,-1.93
25%,-0.68,-0.69,-0.70,-0.69,-0.70,-0.65,-0.90,-0.90
50%,-0.01,-0.04,-0.01,-0.04,-0.04,-0.01,0.17,0.14
75%,0.65,0.68,0.69,0.66,0.69,0.64,0.17,0.65
max,2.48,3.22,3.17,3.34,3.40,3.03,4.42,3.76


## **Section 14 — Final X/y Verification**
Confirm shapes, dtypes, and that no leakage-prone step used test data.
Feature count is read from the actual objects, not hardcoded.

In [15]:
assert X_train.isnull().sum().sum() == 0 and X_test.isnull().sum().sum() == 0
assert list(X_train.columns) == list(X_test.columns)
assert X_train.select_dtypes(include='object').shape[1] == 0

print(f"Final feature count: {X_train.shape[1]}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

Final feature count: 66
X_train: (800, 66), X_test: (200, 66)
y_train: (800,), y_test: (200,)


## **Section 15 — Save Processed Datasets and Artifacts**
Saves model-ready splits for Task 05, plus the fitted scaler and the final
column order (needed later so the prototype/app applies identical encoding
at inference time).

In [16]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
joblib.dump(list(X_train.columns), MODELS_DIR / 'feature_columns.pkl')

print("Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv, scaler.pkl, feature_columns.pkl")

Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv, scaler.pkl, feature_columns.pkl


## Section 16 — Task 03 Summary

| Step | Action | Leakage-safe? |
|---|---|---|
| Missing values | `room_type`: structural "Not Admitted" (pre-split) + mode (train-only, post-split) | Yes |
| Duplicates | Checked at 3 levels — none found | N/A |
| Outliers | IQR-investigated, kept for clinical predictors (documentation only) | Yes (no fit) |
| Cleaning | 16 rows capped, 3 rows swapped (assumed transposition), 212 documented | Yes (deterministic) |
| Target/predictor separation | Dropped identifiers, alt-targets, `appointment_date`; ordinal-encoded target | Yes |
| Split | Stratified 80/20 | — |
| Encoding | One-hot, categories fit on train, aligned to test | Yes |
| Feature engineering | 5 features, fixed clinical thresholds | Yes |
| Feature selection | Correlation computed on `y_train` only | Yes (fixed from earlier full-df version) |
| Scaling | `StandardScaler` fit on `X_train` only | Yes |
| Final feature count | Computed programmatically — see Section 14 | — |

**Note:** the earlier version of this notebook computed feature-selection
correlations and (implicitly) some encoding decisions on the full dataset
before splitting. This version fixes that by deferring every data-dependent
step to after the stratified split and fitting exclusively on `X_train`.